In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import random
import re

# -----------------------------
# 1. Synthetic Dataset Creation
# -----------------------------

spam_keywords = ["free", "offer", "win", "winner", "cash", "prize"]
urgency_keywords = ["urgent", "now", "limited", "hurry", "immediately"]

def generate_email(spam=True):
    text = ""

    if spam:
        # Add spam words
        text += " ".join(random.choices(spam_keywords, k=random.randint(1, 3)))
        # Add urgency
        if random.random() > 0.5:
            text += " " + random.choice(urgency_keywords)
        # Add links
        links = " ".join(["http://spam.com"] * random.randint(1, 3))
        text += " " + links
    else:
        text = "Hello, let's schedule a meeting tomorrow."

    return text.strip()

def extract_features(email):
    email = email.lower()

    # N1: Free/offer words
    has_spam_words = int(any(word in email for word in spam_keywords))

    # N2: Number of links
    link_count = len(re.findall(r'http', email))

    # N3: Message length
    length = len(email.split())

    # N4: Urgency tone
    has_urgency = int(any(word in email for word in urgency_keywords))

    return [has_spam_words, link_count, length, has_urgency]

# Generate dataset
X = []
y = []

for _ in range(1000):
    if random.random() > 0.5:
        email = generate_email(spam=True)
        label = 1  # Spam
    else:
        email = generate_email(spam=False)
        label = 0  # Not spam

    features = extract_features(email)
    X.append(features)
    y.append(label)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

# Normalize length feature
X[:, 2] = X[:, 2] / np.max(X[:, 2])

# -----------------------------
# 2. Model (Your Layer Design)
# -----------------------------

model = models.Sequential([
    # Layer 1 (basic neurons)
    layers.Dense(8, activation='relu', input_shape=(4,)),

    # Layer 2 (pattern combinations)
    layers.Dense(6, activation='relu'),

    # Layer 3 (decision signals)
    layers.Dense(4, activation='relu'),

    # Output layer
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# -----------------------------
# 3. Train Model
# -----------------------------

model.fit(X, y, epochs=10, batch_size=32, validation_split=0.2)

# -----------------------------
# 4. Test Prediction
# -----------------------------

def predict_email(email_text):
    features = np.array([extract_features(email_text)], dtype=np.float32)
    features[:, 2] = features[:, 2] / np.max(X[:, 2])  # normalize

    prob = model.predict(features)[0][0]

    print(f"\nEmail: {email_text}")
    print(f"Spam Probability: {prob:.4f}")

    if prob > 0.5:
        print("🚨 Spam")
    else:
        print("✅ Not Spam")

# Example tests
predict_email("Win free cash now http://spam.com")
predict_email("Hey, are we meeting tomorrow?")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.0000e+00 - loss: 1.0638 - val_accuracy: 0.0000e+00 - val_loss: 0.8937
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0150 - loss: 0.8208 - val_accuracy: 0.0550 - val_loss: 0.7420
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3125 - loss: 0.7029 - val_accuracy: 0.4350 - val_loss: 0.6759
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5050 - loss: 0.6391 - val_accuracy: 0.4350 - val_loss: 0.6240
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5050 - loss: 0.5870 - val_accuracy: 0.4350 - val_loss: 0.5827
Epoch 6/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7688 - loss: 0.5368 - val_accuracy: 1.0000 - val_loss: 0.5288
Epoch 7/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.4768 - val_accuracy: 1.0000 - val_loss: 0.4664
Epoch 8/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.4135 - val_accuracy: 1.0000 - 